In [1]:
import geopandas as gpd
import pandas as pd
import random
from src.space.utils import _to_gpd

## 1, Load Data

In [5]:
#ALL UB Building footprint
building = gpd.read_file('data/input/Single_Buildings.shp')
building.head()

,SITE,FACNAME,PRIMUSE,FLOORS,WIRELESS,STATUS,DEPLOYDATE,DEPLOYSEM,MAPID,MAPAREA,...,POORREPORT,TOTREPORTS,SUMSCORE,FINALSCORE,GLOBALID,CompleteDT,GlobalID_2,Shape_Leng,Shape_Area,geometry
0,NORTH,Ellicott - Red Jacket,Residen,10,56,C,2017-05-15,11,213,4.764184e+06,...,43148.0,0,0,0,680397.302265951,2017-08-31,{BF5D553E-4352-47D3-9CEB-0824FE3FF2EE},1586.042451,43148.187495,"POLYGON Z ((-78.78644 43.00931 0.00000, -78.78..."
1,NORTH,Ellicott - Porter,Residen,10,54,C,2017-05-15,11,212,4.764077e+06,...,30650.0,0,0,0,680404.982751606,2017-08-31,{BC642F74-8AFE-4D3F-AA04-78FF72E8568C},1396.558370,30649.830354,"POLYGON Z ((-78.78654 43.00822 0.00000, -78.78..."
2,NORTH,Alumni Arena Offices,Art/Ath,3,79,C,2016-12-20,3,103,4.763222e+06,...,189152.0,0,0,0,680851.403942639,2017-01-06,{055EEBA7-2EFD-497A-80B8-B59E19A1D595},2099.972835,189151.991484,"POLYGON Z ((-78.78071 43.00086 0.00000, -78.78..."
3,NORTH,Alumni Arena Offices,Art/Ath,3,79,C,2016-12-20,3,103,4.763222e+06,...,189152.0,0,0,0,680851.403942639,2017-01-06,{055EEBA7-2EFD-497A-80B8-B59E19A1D595},2099.972835,189151.991484,"POLYGON Z ((-78.78126 43.00091 0.00000, -78.78..."
4,NORTH,Center For The Arts,Art/Ath,4,49,C,2016-09-01,2,121,4.763307e+06,...,102969.0,2,0,0,680717.917281465,2016-09-20,{D30B407D-1250-40D8-8719-8415F9B1BC2F},1500.758160,102968.782665,"POLYGON Z ((-78.78310 43.00150 0.00000, -78.78..."


In [6]:
#Census Data indicates the number of population we are creating
census = gpd.read_file('data/raw/Ub_census_tract.shp')
census.head()

,GEOID10,NAMELSAD10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,DP0010001,DP0010002,DP0010003,DP0010004,...,DP0210001,DP0210002,DP0210003,DP0220001,DP0220002,DP0230001,DP0230002,Shape_Leng,Shape_Area,geometry
0,36029009110,Census Tract 91.10,4665381.0,231771.0,+43.0013372,-078.7811290,5737,0,0,0,...,2,0,2,0,3,0.0,1.5,0.126128,0.000541,"POLYGON ((-78.78197 42.99595, -78.78202 42.996..."


## 2, Select Buildings

### 2.1 Residential Building

In [7]:
building.PRIMUSE.unique()

array(['Residen', 'Art/Ath', 'Adm/Serv', 'Academic', 'Acad/Res', 'Point',
       'Research', 'Pub Serv'], dtype=object)

In [ ]:
res_building = building[(building.PRIMUSE == 'Residen') | (building.PRIMUSE == 'Acad/Res')]
len(res_building)

In [ ]:
res_building.plot()  

In [ ]:
#create the center point for each residential buildings
res_building['c_point'] = res_building.centroid
res_building.head()

In [ ]:
res_building['long'] = res_building['c_point'].x
res_building['lat'] = res_building['c_point'].y
res_building.head()

In [ ]:
col_list = ['MAPID', 'FACNAME', 'FLOORS', 'long', 'lat']
res_point = res_building.loc[:, col_list].sort_values(by='FACNAME').reset_index(drop = True)
res_point['MAPID'] = res_point.index
res_point

In [ ]:
#res_point = to_GPD(res_point)
#res_point.to_file('res_building_points.shp')

### 2.2 Class Building 
Buildings with type of acdemic and research

In [ ]:
class_building = building[(building.PRIMUSE == 'Academic') | (building.PRIMUSE == 'Research') | (building.PRIMUSE == 'Art/Ath')]
len(class_building)

In [ ]:
class_building['c_point'] = class_building.centroid
class_building.head()

In [ ]:
class_building['long'] = class_building['c_point'].x
class_building['lat'] = class_building['c_point'].y
class_building.head()

In [ ]:
class_point = class_building.loc[:, col_list].sort_values(by='FACNAME').reset_index(drop = True)
class_point['MAPID'] = class_point.index
class_point

In [ ]:
class_point = to_geo_df(class_point)

In [ ]:
building.plot()

In [ ]:
class_point.plot()  

In [ ]:
class_point.to_csv('class_building_points.csv')

In [ ]:
class_point.to_file('class_point.shp')

### 3, Create Population

In [ ]:
popNum = census.DP0010001.values[0]
print("creating", popNum, "population")

In [ ]:
res_point.FLOORS.unique()

In [ ]:
res_point.FLOORS.sum()

In [ ]:
5737 / res_point.FLOORS.sum()

In [ ]:
popNum / len(res_point)

In [ ]:
res_point

In [ ]:
#Random int generator based on normal distribution
def GenBoundedRandomNormal(meanVal, stdDev, lowerBound, upperBound):
    aRand = random.gauss(meanVal,stdDev) # could also use: normalvariate()but gauss () is slightly faster.
    
    while (aRand < lowerBound or aRand > upperBound):
        aRand = random.gauss(meanVal,stdDev)
    return aRand

In [ ]:
def CreateStudents(data, class_point, popNum, id_list, age_list, class_loc, x_list, y_list):
    
    print("Res Building", data.MAPID,"Floor", data.FLOORS)
    
    
    def check_number(f):
        if f == 1:
            return 32
        else: return 32 * f
    
    newpop = check_number(data.FLOORS)
    #print(newpop)

    if (len (id_list) + newpop) > popNum:
        for i in range (popNum - len (id_list)):
            in_floor = random.randint(1, data.FLOORS)#ramdom assign floor
            pid = 'b' + str(data.MAPID) + str(in_floor) + str(i)

            id_list.append(pid)
            age_list.append(int(GenBoundedRandomNormal(20, 4, 17, 26)))
            #class_loc.append(random.choice(class_point.MAPID))
            class_loc.append(random.choice(class_point.id))
            y_list.append(data.long)
            x_list.append(data.lat)
        
    else:
        if data.FLOORS > 1: #40 people each floor
            for floor in range(data.FLOORS):
                in_floor  = floor + 1
                for i in range(0, 32):
                    pid = 'b' + str(data.MAPID) + str(in_floor) + str(i)

                    id_list.append(pid)
                    age_list.append(int(GenBoundedRandomNormal(20, 4, 17, 26)))
                    class_loc.append(random.choice(class_point.id))
                    y_list.append(data.long)
                    x_list.append(data.lat)
                #temp_df = pd.DataFrame(list(zip(id_list, age_list, class_loc, y_list, x_list)),columns =['id', 'age', 'class', 'long', 'lat'])
                #pop_df = pd.concat([pop_df, temp_df]) 
                #print(len(temp_df))
                #pop_df.append(temp_df)

        elif data.FLOORS == 1: #80 people per floor
            for i in range(0, 32):
                pid = 'b' + str(data.MAPID) + str(data.FLOORS) + str(i)
                id_list.append(pid)
                age_list.append(int(GenBoundedRandomNormal(20, 4, 17, 26)))
                class_loc.append(random.choice(class_point.id))
                y_list.append(data.long)
                x_list.append(data.lat)
    
    return "Res Building", data.MAPID, "Done!"

In [ ]:
uid_l = []
a_l = []
c_l = []
x_l = []
y_l = []

res_point.apply(CreateStudents, args=(class_point, popNum, uid_l , a_l, c_l, x_l, y_l),axis=1)

In [ ]:
temp_df = pd.DataFrame(list(zip(uid_l, a_l, c_l, x_l, y_l)),columns =['id', 'age', 'class', 'long', 'lat'])
temp_df

In [ ]:
temp_df.to_csv('ub_pop.csv')